## Setup

This notebook walks through `pydantic-ai`'s core primitives: schemas, agents, structured output, dependency injection, the five output modes, run methods, and a function tool. Cells build on each other; run them in order.

Output uses `rprint` from `rich`, which formats Python objects (models, dicts, exceptions) more readably than the built-in `print`.

In [39]:
from rich.markdown import Markdown
from rich import print as rprint

rprint(Markdown("# Introduction to pydantic-ai"))
rprint({"user": "Alice", "question": "What is this?"})

Introduction to pydantic-ai

{'user': 'Alice', 'question': 'What is this?'}

## Pydantic models

Three schemas the rest of the notebook reuses:

- **`Question`** — the input shape, with length constraints.
- **`Citation`** — a pointer to a sentence or paragraph in a source document.
- **`Answer`** — the output shape: text, optional citations, a confidence score, and an escalation flag.

`Answer` declares two validators:

- A field-level validator that rejects empty citation quotes.
- A model-level validator that rejects low-confidence answers whose text doesn't hedge.

The same `ValidationError` fires whether the input came from your own code or from an LLM's tool call.

In [40]:
from typing import Annotated, Literal
from pydantic import (
    BaseModel, ConfigDict, Field, EmailStr, SecretStr, PositiveInt,
    StrictInt, ValidationError, field_validator,
)

class AgeLax(BaseModel):        # default: coerces "30" -> 30
    age: int
class AgeStrictType(BaseModel): # StrictInt: refuses the string
    age: StrictInt
class AgeStrictField(BaseModel):# Field(strict=True): strict + a bound
    age: Annotated[int, Field(strict=True, ge=18)]

print("lax        :", AgeLax(age="30").age)                       # 30
for M in (AgeStrictType, AgeStrictField):
    try:
        M(age="30")
    except ValidationError as e:
        print(f"{M.__name__:15}: rejected '30' -> {e.errors()[0]['type']}")

class Signup(BaseModel):
    model_config = ConfigDict(
    # Refuse type coercion: an int field accepts 30 but rejects "30" / 30.0.
    # Applies to every field in this model.
    strict=True,

    # Reject unknown keys. Passing a field that isn't declared raises a ValidationError (default is "ignore").
    extra="forbid",

    # Re-run validation when you assign to a field AFTER construction
    # (obj.age = "x"). Without this, only __init__ is validated and later
    # assignments bypass all checks.
    validate_assignment=True,

    # Strip leading/trailing whitespace from every str field before
    # validation, so "  alice@example.com  " becomes "alice@example.com".
    str_strip_whitespace=True,

    # frozen=True
)
    email: EmailStr
    password: SecretStr
    age: PositiveInt
    tier: Literal["basic", "pro", "enterprise"] = "basic"

    @field_validator("password")
    @classmethod
    def strong_enough(cls, v: SecretStr) -> SecretStr:
        if len(v.get_secret_value()) < 6:
            raise ValueError("password too short")
        return v


u = Signup(email="  alice@Example.com ", password="apapapappapap231213APAPP", age=30, tier="pro")
print("\nemail      :", u.email)
print("password   :", u.password)
print("model dump :", u.model_dump())

# --- the same data, three parse outcomes ---------------------------------
data = {"email": "bob@example.com", "password": "secret1", "age": "41"}
try:
    Signup(**data)                                  # strict model: "41" rejected
except ValidationError as e:
    print("\nstrict constructor:", e.errors()[0]["type"], "on", e.errors()[0]["loc"])
print("lax parse  :", Signup.model_construct(**data).age,  # skips validation
      "(model_construct: no coercion & no checks)")

# strictness can also be decided per-call:
class Item(BaseModel):
    qty: int
print("per-call   :", Item.model_validate({"qty": "5"}).qty,            # 5  (lax)
      "/ strict ->", end=" ")
try:
    Item.model_validate({"qty": "5"}, strict=True)
except ValidationError as e:
    print(e.errors()[0]["type"])                                        # int_type

# frozen + assignment guard
try:
    u.age = "fifty"          # validate_assignment + strict catches this post-build
except ValidationError as e:
    print("\nassignment :", e.errors()[0]["type"], "(guarded after construction)")

lax        : 30
AgeStrictType  : rejected '30' -> int_type
AgeStrictField : rejected '30' -> int_type

email      : alice@example.com
password   : **********
model dump : {'email': 'alice@example.com', 'password': SecretStr('**********'), 'age': 30, 'tier': 'pro'}

strict constructor: int_type on ('age',)
lax parse  : 41 (model_construct: no coercion & no checks)
per-call   : 5 / strict -> int_type

assignment : int_type (guarded after construction)


In [41]:
from __future__ import annotations

from typing import Annotated, Literal

from pydantic import BaseModel, Field, ValidationError, field_validator, model_validator


class Question(BaseModel):
    text: Annotated[str, Field(min_length=3, max_length=2000)]
    user_id: str | None = Field(
        default=None,
        description="Optional user identifier for the session.",
    )


class Citation(BaseModel):
    """A pointer to a sentence or paragraph in a source document."""

    doc_id: Annotated[str, Field(min_length=1, max_length=100)]
    quote: Annotated[str, Field(min_length=1, max_length=500)]


class Answer(BaseModel):
    """A structured assistant response.

    `risk_flag` lets the assistant signal escalation needs (out-of-scope
    or sensitive queries, urgent issues) without burying that signal in
    the response text.
    """

    text: Annotated[str, Field(min_length=1, max_length=4000)]
    citations: list[Citation] = Field(default_factory=list)
    confidence: Annotated[float, Field(ge=0.0, le=1.0)]
    risk_flag: Literal["none", "escalate", "urgent"] = "none"

    @field_validator("citations")
    @classmethod
    def reject_empty_citation_quotes(cls, v: list[Citation]) -> list[Citation]:
        if any(not c.quote.strip() for c in v):
            raise ValueError("citation quotes must not be empty or whitespace")
        return v

    @model_validator(mode="after")
    def low_confidence_must_admit_uncertainty(self) -> "Answer":
        text_l = self.text.lower()
        if self.confidence < 0.4 and "not sure" not in text_l and "may" not in text_l:
            raise ValueError(
                "confidence < 0.4 — answer text must signal uncertainty "
                '(e.g. contain "not sure" or "may")'
            )
        return self

## Validators run at construction time

Construct `Answer` directly:

- The first call satisfies both validators.
- The second deliberately trips `low_confidence_must_admit_uncertainty` (confidence `0.1` paired with confident-sounding text).

The `ValidationError` printed in the second case is exactly what an agent receives when its draft is rejected — and feeds back to the model on retry.

In [42]:
# Happy path
ok = Answer(text="Standard shipping arrives in 3-5 business days.", confidence=0.95)
rprint("valid:", ok)

# Sad path — confidence is low but the text doesn't hedge
try:
    Answer(text="It costs 5 dollars.", confidence=0.1)
except ValidationError as exc:
    rprint("\nValidationError raised, as expected:")
    rprint(exc)

valid:
Answer(text='Standard shipping arrives in 3-5 business days.', citations=[], confidence=0.95, risk_flag='none')

ValidationError raised, as expected:

1 validation error for Answer
  Value error, confidence < 0.4 — answer text must signal uncertainty (e.g. contain "not sure" or "may") 
[type=value_error, input_value={'text': 'It costs 5 dollars.', 'confidence': 0.1}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

## Typed configuration with `pydantic-settings`

`BaseSettings` reads from environment variables and `.env`, validates types, and raises on missing required fields.

Benefits over scattered `os.getenv(...)` calls:

- Types are enforced — `int`, `Literal`, `Path`, etc.
- Missing required values raise *at construction*.

In [43]:
from pathlib import Path

PROJECT_ROOT_PATH = Path.cwd().parent.parent
rprint(PROJECT_ROOT_PATH)

/Users/dan/Things/job/datasentics/projects/kbc_workshop/pydantic_ai_workshop

In [44]:
from pathlib import Path

from pydantic_settings import BaseSettings, SettingsConfigDict

_ENV_FILE = PROJECT_ROOT_PATH / ".env"


class Settings(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=_ENV_FILE,
        env_file_encoding="utf-8",
        extra="ignore",
    )

    anthropic_api_key: SecretStr = Field(
        ...,
        description="Workshop-allocated Anthropic API key. Required.",
    )
    model_id: str = Field(
        default="claude-haiku-4-5-20251001",
        description="Default model id. Haiku is cheap and fast for the demos.",
    )
    log_level: Literal["DEBUG", "INFO", "WARNING", "ERROR"] = "INFO"


settings = Settings()  # type: ignore[call-arg]

rprint("model_id  :", settings.model_id)
rprint("log_level :", settings.log_level)
rprint("api key   :", settings.anthropic_api_key)

model_id  : claude-haiku-4-5-20251001

log_level : INFO

api key   : **********

## Build the model

`AnthropicModel` wraps the provider (which holds credentials) and the model id. Every `Agent(...)` constructor in the rest of the notebook receives this `model` instance.

In [45]:
from pydantic_ai.models.anthropic import AnthropicModel
from pydantic_ai.providers.anthropic import AnthropicProvider

model = AnthropicModel(
    settings.model_id,
    provider=AnthropicProvider(api_key=settings.anthropic_api_key.get_secret_value()),
)

## Optional — log HTTP requests

Uncomment to log every HTTP request the Anthropic SDK sends. The output is going to be very verbose.

In [46]:
# import logging
# logging.basicConfig()
# logging.getLogger("anthropic").setLevel(logging.DEBUG)
# logging.getLogger("httpx").setLevel(logging.DEBUG)

## Smallest possible agent

A model plus an instruction string. With no `output_type` argument the default applies (`str`), so `result.output` is the plain text the model returned.

In [47]:
from pydantic_ai import Agent

trivial = Agent(model, instructions="You are a helpful tutor. Be brief.")

result = await trivial.run("What is 2 + 2?")
rprint("type(result.output):", type(result.output).__name__)
rprint("result.output:", result.output)

type(result.output): str

result.output: 2 + 2 = 4

In [ ]:
for message in result.all_messages():
    rprint(message)

```plain

ModelRequestNode [ UserPromptNode ]
      │                                          
      ▼                                        
HandleModelResponseNode [ TextPart ]                         
      │                                          
      └──► End
```

## Structured output with `output_type=Answer`

Setting `output_type=Answer` makes `pydantic-ai` validate the LLM's reply against the schema before returning. If validation fails the agent retries, feeding the validation error back to the model so it can correct the output.

`result.output` is then a typed `Answer` instance: attribute access (`.confidence`, `.risk_flag`, `.citations`) works directly without `json.loads` or `dict` indexing.

In [48]:
from pydantic_ai import RunContext

assistant = Agent[str, Answer](
    model,
    output_type=Answer,
    deps_type=str,
    instructions=(
        "You are a helpful customer support assistant. "
        "Always answer in the structured Answer format. "
        "If you are not sure, set confidence below 0.4 and say so. "
        "If the question is sensitive or out of scope, "
        "set risk_flag='escalate'. "
        "If the question concerns an urgent issue, "
        "set risk_flag='urgent'."
    ),
)


@assistant.instructions
def greet_by_name(ctx: RunContext[str]) -> str:
    return (
        f"The user in this session is **{ctx.deps}**. "
        "Address them by first name once at the start of your reply."
    )

In [49]:
result = await assistant.run("What is your return policy?", deps="Alice")

rprint("isinstance(result.output, Answer):", isinstance(result.output, Answer))
rprint("text:", result.output.text)
rprint("confidence:", result.output.confidence)
rprint("risk_flag:", result.output.risk_flag)
rprint("citations:", result.output.citations)

isinstance(result.output, Answer): True

text: Hi Alice,

I'm not sure about the specific details of our return policy, as I don't have access to that documentation in my 
current resources. 

To get accurate information about our return policy, I'd recommend:

1. **Visiting our website** - Return policy information is typically available in the footer or "Help" section
2. **Contacting our customer support team directly** - They can provide detailed information about timeframes, 
conditions, and the return process
3. **Checking your order confirmation email** - Return policy details are often included there

If you have a specific product or situation you need help with regarding a return, please let me know the details, 
and I'll do my best to assist you further or escalate to the appropriate team.

Is there anything else I can help you with?

confidence: 0.25

risk_flag: none

citations:
[]

In [ ]:
for message in result.all_messages():
    rprint(message)

```plain
ModelRequestNode [ UserPromptNode ]
      │                                          
      ▼                                        
HandleModelResponseNode [ ToolCallPart - final_result ]                         
      │
      ▼
ModelRequestNode [ ToolReturnPart - final_result ]
      │                                                                                   
      └──► End
```

## Dependency injection via `deps_type`

The agent declared `deps_type=str`, so every call must pass `deps=`. Inside the agent, `RunContext` exposes the value as `ctx.deps`.

The `@assistant.instructions` decorator below registers a *per-run* callback. It receives the `RunContext` for the current call and returns a string fragment that joins onto the static instructions.

In [50]:
for user in ("Alice", "Bob"):
    result = await assistant.run("Can you remind me how shipping works?", deps=user)
    rprint(f"--- deps={user!r} ---")
    rprint(result.output.text)
    rprint()

--- deps='Alice' ---

Hi Alice, I'd be happy to help, but I'm not sure which specific shipping process you're referring to. I don't have 
access to your company's shipping policies and procedures in our current conversation. 

To give you accurate information, could you provide more context about:

1. **Which company or service** you're asking about (e.g., your online store, a specific retailer, or shipping 
carrier)
2. **What specific shipping details** you're interested in (e.g., how orders are processed, delivery timeframes, 
shipping costs, tracking, return shipping)

Once you share those details, I'll be able to give you a clear reminder of how shipping works for your situation.

--- deps='Bob' ---

Hi Bob, I'm not sure which specific shipping information you're looking for. I don't have access to documented 
shipping policies in our current conversation. 

To help you better, could you clarify what you'd like to know about shipping?

1. Are you asking about shipping for a **specific order** you've placed?
2. Are you asking about our **general shipping policies** (delivery times, costs, coverage areas)?
3. Are you asking about **how to arrange a shipment** for something?
4. Are you asking about a **return or replacement shipment**?

Once you provide more details, I'll be able to give you precise information about our shipping process.

## Output modes side-by-side

Five `output_type` shapes against the same `CityLocation` schema and the same prompt:

| Mode | Notes |
|---|---|
| `str` | Plain text. No schema. |
| `ToolOutput` | Default. Schema-validated via a synthetic output tool. |
| `NativeOutput` | Provider-side JSON mode. |
| `PromptedOutput` | Schema described in the prompt (fallback for weaker models). |
| `TextOutput(parser)` | Free-form text parsed by your own function. |

The next cell runs all five and prints `type(result.output)` for each.

In [51]:
from pydantic_ai import NativeOutput, PromptedOutput, TextOutput, ToolOutput


class CityLocation(BaseModel):
    """Tiny structured shape used by every demo agent below."""

    city: str
    country: str


PROMPT = "Where were the 2012 Olympics held?"


def parse_csv(text: str) -> CityLocation:
    """Parse 'city, country' text into a CityLocation."""
    parts = [p.strip() for p in text.split(",", 1)]
    if len(parts) != 2:
        raise ValueError(f"expected 'city, country', got {text!r}")
    return CityLocation(city=parts[0], country=parts[1])


_instructions = (
    "Answer the user's geography question. When asked for a CSV string, "
    "respond with exactly 'city, country' and nothing else."
)

mode_agents: dict[str, Agent] = {
    "str (raw text)": Agent(model, output_type=str, instructions=_instructions),
    "ToolOutput (default)": Agent(
        model, output_type=ToolOutput(CityLocation), instructions=_instructions
    ),
    "NativeOutput (provider-native JSON)": Agent(
        model, output_type=NativeOutput(CityLocation), instructions=_instructions
    ),
    "PromptedOutput (no-protocol fallback)": Agent(
        model, output_type=PromptedOutput(CityLocation), instructions=_instructions
    ),
    "TextOutput (custom parser)": Agent(
        model, output_type=TextOutput(parse_csv), instructions=_instructions
    ),
}

## Run each mode

Same call shape for every entry: `await agt.run(PROMPT)`. The output type and value differ depending on which `output_type=` the agent was built with.

In [52]:
rprint(f"Prompt: {PROMPT}\n")
for label, agt in mode_agents.items():
    result = await agt.run(PROMPT)
    rprint(f"--- {label} ---")
    rprint(f"type:  {type(result.output).__name__}")
    rprint(f"value: {result.output!r}\n")

Prompt: Where were the 2012 Olympics held?

--- str (raw text) ---

type:  str

value: 'London, United Kingdom'

--- ToolOutput (default) ---

...

type:  CityLocation

value: CityLocation(city='London', country='United Kingdom')

--- TextOutput (custom parser) ---

type:  CityLocation

value: CityLocation(city='The 2012 Olympics were held in London', country='United Kingdom.')

## Run methods

Three ways to invoke the same agent:

- `agent.run_sync(...)` — blocks the calling thread.
- `await agent.run(...)` — async coroutine, the pydantic-ai default.
- `async with agent.run_stream(...) as result` — yields text deltas as the model emits them.

(`agent.iter(...)` for node-by-node inspection is covered in a later notebook.)

In [53]:
# run_sync — blocking
import nest_asyncio
nest_asyncio.apply()

sync_result = assistant.run_sync("Summarize the return policy in one sentence.", deps="Alice")
rprint("run_sync   ->", sync_result.output.text[:120], "...")


run_sync   -> Alice, I'm not sure how to help with this request. I don't have access to any return policy documents
or information. To ...

In [54]:
# await run — async, the default in async code
async_result = await assistant.run("Summarize the return policy in one sentence.", deps="Alice")
rprint("await run  ->", async_result.output.text[:120], "...")

await run  -> I'm not sure what return policy you're referring to, as I don't have access to any return policy 
documentation. To help  ...

In [55]:
# run_stream — token-by-token. Use a str-output agent so the deltas print cleanly.
text_agent = Agent(
    model,
    output_type=str,
    instructions="You are a helpful support assistant. Give a friendly two-sentence answer.",
)

print("run_stream ->", end=" ")
async with text_agent.run_stream("How does standard shipping work?") as result:
    async for chunk in result.stream_text(delta=True):
        print(chunk, end="", flush=True)
print()

run_stream -> Standard shipping typically takes 5-7 business days for delivery after your order is placed, and it's usually the most affordable shipping option available. Your package will be transported through our carrier network and tracked so you can monitor its progress along the way!


## Function tools

A plain Python function. Registering it on an agent (next cell) turns the function's signature and docstring into the tool schema the LLM sees. The model decides when to call the tool; `pydantic-ai` parses the call, runs the function, and feeds the result back into the conversation.

In [56]:
_PRICE_LIST: dict[str, float] = {
    "basic_plan": 9.99,
    "pro_plan": 29.99,
    "enterprise_plan": 99.99,
    "addon_storage": 4.99,
    "addon_priority_support": 19.99,
}


def lookup_price(item_name: str) -> dict:
    """Look up a product price by canonical name. Returns USD amount or an error."""
    key = item_name.strip().lower()
    if key not in _PRICE_LIST:
        return {"error": "unknown item", "known": sorted(_PRICE_LIST)}
    return {"item_name": key, "price_usd": _PRICE_LIST[key]}

## `event_stream_handler=` — observe a run

Pass an async callback as `event_stream_handler=` and the agent will fire it for every event, such as: tool calls, tool results and model deltas alongside the normal `await agent.run(...)` return value.

The handler below prints each tool call and its result as the loop runs.

In [57]:
from collections.abc import AsyncIterable

from pydantic_ai.messages import (
    AgentStreamEvent,
    FunctionToolCallEvent,
    FunctionToolResultEvent,
)


async def print_tool_events(
    ctx: RunContext[None],
    events: AsyncIterable[AgentStreamEvent],
) -> None:
    async for event in events:
        if isinstance(event, FunctionToolCallEvent):
            print(f"[tool: {event.part.tool_name}({event.part.args})]")
        elif isinstance(event, FunctionToolResultEvent):
            print(f"[result: {event.part.content!r}]")


tool_agent: Agent[None, Answer] = Agent(
    model,
    output_type=Answer,
    instructions=(
        "You are a helpful assistant. Use the lookup_price tool when asked "
        "about a specific product price. Always answer in Answer format and "
        "cite the tool's return as a citation with doc_id='pricing'."
    ),
    tools=[lookup_price],
)

result = await tool_agent.run(
    "What does the pro_plan cost?",
    event_stream_handler=print_tool_events,
)
print(f"\nfinal: {result.output.text}")
print(f"conf : {result.output.confidence}")

[tool: lookup_price({"item_name": "pro_plan"})]
[result: {'item_name': 'pro_plan', 'price_usd': 29.99}]

final: The pro_plan costs $29.99 USD.
conf : 0.95


## Recap

- Pydantic models declare the input/output shapes. Validators run on construction and during agent output validation.
- `pydantic-settings` replaces scattered `os.getenv(...)` calls with a typed `Settings` object.
- `Agent` ties together a model, instructions, an output schema, and (optionally) typed dependencies.
- `output_type=` chooses how the model's reply is decoded. `ToolOutput` is the default; the other four cover provider quirks and custom parsers.
- `deps_type=` + `@agent.instructions` keep per-run state out of the static prompt.
- Run methods: `run` for async code, `run_stream` for token-by-token UIs, `run_sync` for demos.
- `event_stream_handler=` exposes the agent's internal events without consuming the result.